# 02 - RoPE 位置编码 (AI Infra 视角)

本节从 **工程实现** 角度理解 RoPE：
- 核心原理 (不讲复数推导)
- 预计算与缓存
- 长度外推 (NTK-aware, YaRN)
- 实现细节

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

## 1. 为什么需要位置编码 (30秒版)

**问题**: Attention 是位置无关的

```
输入 [A, B, C] 和 [C, B, A] 
对 Attention 来说是一样的
但 "我爱你" 和 "你爱我" 意思完全不同!
```

**解决**: 位置编码让模型知道每个 token 在哪里

## 2. 位置编码的演进

| 方法 | 原理 | 问题 | 使用模型 |
|------|------|------|----------|
| 绝对位置 | 学习每个位置的向量 | 长度受限 | GPT-2 |
| 正弦位置 | sin/cos 函数 | 信息逐层衰减 | 原版Transformer |
| ALiBi | 注意力 bias | 简单但效果一般 | BLOOM |
| **RoPE** | 旋转 Q, K | **相对位置 + 外推性** | **LLaMA, GPT-4** |

### RoPE 的核心思想

```
不是把位置编码"加"到向量上
而是根据位置"旋转"Q 和 K 向量

关键性质: 旋转后的 Q·K 点积只依赖相对位置
```

## 3. RoPE 实现

### 核心公式 (简化版)

```
把向量分成对: [x0, x1], [x2, x3], ...
每对用不同频率旋转:

  [x0']   [cos(θ)  -sin(θ)] [x0]
  [x1'] = [sin(θ)   cos(θ)] [x1]

θ = position * base^(-2i/d)
```

In [ ]:
def precompute_rope(seq_len, head_dim, base=10000):
    """
    预计算 RoPE 的 cos 和 sin
    
    关键: 这个只需要计算一次，推理时直接查表
    """
    # 每个维度对的频率: 1 / (base^(2i/d))
    dim_indices = torch.arange(0, head_dim, 2).float()
    inv_freq = 1.0 / (base ** (dim_indices / head_dim))
    
    # 每个位置的角度
    positions = torch.arange(seq_len).float()
    angles = torch.outer(positions, inv_freq)  # (seq_len, head_dim/2)
    
    cos = angles.cos()  # (seq_len, head_dim/2)
    sin = angles.sin()
    
    # 添加 batch 和 head 维度: (1, seq_len, 1, head_dim/2)
    return cos[None, :, None, :], sin[None, :, None, :]


def apply_rope(x, cos, sin):
    """
    应用 RoPE 旋转
    
    x: (batch, seq_len, n_heads, head_dim)
    """
    d = x.shape[-1] // 2
    x1, x2 = x[..., :d], x[..., d:]  # 分成两半
    
    # 旋转
    y1 = x1 * cos + x2 * sin
    y2 = x1 * (-sin) + x2 * cos
    
    return torch.cat([y1, y2], dim=-1)


# 测试
seq_len, head_dim = 16, 64
cos, sin = precompute_rope(seq_len, head_dim)
print(f"cos shape: {cos.shape}")
print(f"sin shape: {sin.shape}")

x = torch.randn(2, seq_len, 8, head_dim)  # (batch, seq, heads, dim)
y = apply_rope(x, cos, sin)
print(f"输入: {x.shape} -> 输出: {y.shape}")

## 4. 预计算与缓存

**工程要点**: cos/sin 只依赖位置和维度，可以预计算

```python
# nanochat/gpt.py
class GPT(nn.Module):
    def __init__(self, config):
        # 预计算 (10x 最大长度，防止超出)
        self.rotary_seq_len = config.sequence_len * 10
        cos, sin = self._precompute_rotary_embeddings(self.rotary_seq_len, head_dim)
        
        # 存为 buffer (不是参数，不需要梯度)
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
    
    def forward(self, idx):
        T = idx.size(1)
        # 按需截取
        cos_sin = self.cos[:, :T], self.sin[:, :T]
```

In [ ]:
# 可视化不同维度的频率
seq_len = 100
head_dim = 64
cos, sin = precompute_rope(seq_len, head_dim)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
for i in [0, 8, 16, 24, 31]:
    plt.plot(cos[0, :, 0, i].numpy(), label=f'dim {i*2}')
plt.xlabel('Position')
plt.ylabel('cos(θ)')
plt.title('低维变化快 (高频)，高维变化慢 (低频)')
plt.legend()

plt.subplot(1, 2, 2)
plt.imshow(cos[0, :, 0, :].numpy().T, aspect='auto', cmap='RdBu')
plt.xlabel('Position')
plt.ylabel('Dimension')
plt.title('RoPE cos 值')
plt.colorbar()

plt.tight_layout()
plt.show()

## 5. 长度外推 (重要!)

**问题**: 训练时 seq_len=4096，推理时想用 seq_len=32768

### 方法 1: Position Interpolation (简单)

```python
# 把 [0, 32768] 压缩到 [0, 4096]
scale = train_len / target_len  # 4096/32768 = 0.125
positions = positions * scale
```

问题: 分辨率降低

### 方法 2: NTK-aware Scaling

```python
# 调整 base 而不是 position
scale = target_len / train_len
base_new = base * (scale ** (head_dim / (head_dim - 2)))
```

### 方法 3: YaRN (推荐)

结合多种技术，效果最好

In [ ]:
def precompute_rope_ntk(seq_len, head_dim, base=10000, scale=1.0):
    """
    NTK-aware RoPE: 通过调整 base 来外推
    """
    # 调整 base
    if scale != 1.0:
        base = base * (scale ** (head_dim / (head_dim - 2)))
    
    dim_indices = torch.arange(0, head_dim, 2).float()
    inv_freq = 1.0 / (base ** (dim_indices / head_dim))
    positions = torch.arange(seq_len).float()
    angles = torch.outer(positions, inv_freq)
    
    return angles.cos()[None, :, None, :], angles.sin()[None, :, None, :]

# 对比
train_len = 4096
target_len = 16384
scale = target_len / train_len

print(f"训练长度: {train_len}")
print(f"目标长度: {target_len}")
print(f"缩放因子: {scale}x")
print(f"\nNTK-aware 调整 base: 10000 -> {10000 * (scale ** (64 / 62)):.0f}")

## 6. 推理优化: KV Cache 中的 RoPE

**关键**: RoPE 在 KV Cache 场景下的处理

```
Prefill 阶段:
  - 计算 prompt 的 Q, K, V
  - 用 position [0, 1, 2, ...] 的 cos/sin
  - K, V 存入 cache

Decode 阶段:
  - 每次只有 1 个新 token
  - 用 position [current_pos] 的 cos/sin
  - 关键: 位置要对齐!
```

In [ ]:
# nanochat 的 KV Cache 位置偏移
kv_cache_example = '''
# nanochat/gpt.py forward()

# 如果有 KV cache，需要偏移位置
T0 = 0 if kv_cache is None else kv_cache.get_pos()

# 截取对应位置的 cos/sin
cos_sin = self.cos[:, T0:T0+T], self.sin[:, T0:T0+T]

# 例如:
# Prefill: T0=0, T=100  -> 用 [0:100]
# Decode 1: T0=100, T=1 -> 用 [100:101]
# Decode 2: T0=101, T=1 -> 用 [101:102]
'''
print(kv_cache_example)

## 7. 面试常见问题

### Q1: RoPE 和绝对位置编码的区别?

**答**:
- 绝对位置: 加到输入上，`x + pos_emb`
- RoPE: 旋转 Q 和 K，点积自带相对位置信息
- RoPE 优势: 相对位置、长度外推、不增加参数

---

### Q2: RoPE 为什么只对 Q 和 K 应用，不对 V?

**答**:
- 位置信息只需要影响 **注意力权重** (Q·K)
- V 是实际的值，不需要位置信息
- 对 V 应用会浪费计算

---

### Q3: RoPE 的 base=10000 是什么意思?

**答**:
- 控制频率的基数
- `inv_freq = 1 / (base^(2i/d))`
- base 越大，频率越低，能表示的相对距离越远
- LLaMA 2 用 10000，LLaMA 3 用 500000 (支持更长上下文)

---

### Q4: 如何让模型支持比训练时更长的序列?

**答**:
1. **Position Interpolation**: 压缩位置索引
2. **NTK-aware**: 调整 base
3. **YaRN**: 结合多种技术
4. 通常需要少量 fine-tune

---

### Q5: RoPE 的计算开销?

**答**:
- 很小: 只是逐元素乘法
- cos/sin 预计算，推理时查表
- 相比 Attention 本身，开销可忽略

## 8. 总结速查表

| 主题 | 要点 |
|------|------|
| **核心思想** | 旋转 Q, K 向量编码位置 |
| **优势** | 相对位置、长度外推、无额外参数 |
| **预计算** | cos/sin 只依赖位置，提前算好 |
| **只对 Q, K** | V 不需要位置信息 |
| **KV Cache** | 注意位置偏移: `cos[:, T0:T0+T]` |
| **长度外推** | NTK-aware 调整 base |

### nanochat 实现

```python
# 预计算
cos, sin = precompute_rope(seq_len, head_dim, base=10000)

# 应用 (在 Attention 里)
q = apply_rope(q, cos, sin)
k = apply_rope(k, cos, sin)
# v 不变!
```